# IRS PDF Form Build Keys

------------------
## form1065_filler.py
Python class that fills an IRS Form 1065 PDF (U.S. Return of Partnership Income)
from a plain Python dict.


## Workflow: FormKey Mapping

#### A. Gen Key Maps (irs.irsForms)

1. genTestKeyDict
    - irsForm -> fldDict [Field Dict]
    - iraForm -> testDict
    - OUT:
        - testDict
        - FILE: pdfFill(testDict) -> irsForms/<fm>-IRS.pdf.  [visual of forms with F#]
5. genIRSMap
    - fldDict -> keyDict -> irsForms/<fm>-IRS.json [map <fldName> -> F# ]

#### B. Gen IRS Report (irs.pdfFill)

1. gen_glTaxDict
    - glDict -> glKeys. [ Core of IRS Tax mapping]
    - glDict X glKeys -> glTaxDict -> irsForms/<fm>_glKey.json [<fldName -> glDict.alues]
    - glTaxDict X keyDict -> fmDict  [form values: fmKey -> glDict.values]
1. FILE: pdfFill(fmDict) ->  YE_Tax_Report/<fm>-IRS.pdf

------
## Build Key

- input IRS_Forms/formFN.pdf
- get all fields into Dict
- save 

In [1]:
# Load bookkeeping services : llc, coa, 
import os
from ledger.LLC import LLC
from pathlib import Path
import datetime
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

from ledger.llcCOA import ChartOfAccounts
from util.utilEditSession import utilEditSession
import pandas as pd



top = Path.cwd().parents[2]
llc = LLC('WBGroupLLC',debug=False, top=top)  # debug='details'
# Load Latest/YE Bank Stmt
llc._Bank()
# Chart of accounts
coa = ChartOfAccounts(llc)
# Edit Session

In [36]:
from typing import Dict, List, Optional, Tuple

from irs.Form1065 import Form1065
fObj = Form1065(llc, verbose=True)

class xxx:
    def foo(cls, fObj):
        self = fObj
        is_data=None
        bs_data=None
        tstFld = 'f20'
        
        # for verbose test progress
        # ── Step A: base dict — all fielsel
        nSpaceDict = self._buildNSpace()
        fillDict = self._defaultFill(nSpaceDict)
        return fillDict

        # ── Step B: build reverse index logicalKey → fID ──────────────────
        lk_to_fid: Dict[str, str] = {
            fd["logicalKey"]: fid
            for fid, fd in fillDict.items()
            if fd["logicalKey"]
        }
        return lk_to_fid

        # ── Step C: load official IS / BS / owners from llcFinancialReport ─
        owners_agg: Dict = {
            "count":              "0",
            "cash_contributions": 0.0,
            "distributions_cash": 0.0,
        }
        if is_data is None or bs_data is None:
            td = self._resolveTaxData()
            if is_data is None:
                is_data = td.get("is_data", {})
            if bs_data is None:
                bs_data = td.get("bs_data", {})
            owners_agg = td.get("owners", owners_agg)

                
        # ── Step D: load entity / F1065 profile ───────────────────────────
        entity, f1065_data = self._loadProfile()

        # ── Step E: source map ────────────────────────────────────────────
        src_map: Dict = {
            "entity":       entity,
            "F1065":        f1065_data,
            "IS":           is_data   or {},
            "BS":           bs_data   or {},
            "owners":       owners_agg,
            "schB_default": {"No": "/2", "Yes": "/1"},
        }
        if self.verbose:
            print(f"--- {self.oID} buildFillDict StepE verbose{self.verbose}, testFld:{tstFld}, is_data:{len(is_data)}, bs_data:{len(bs_data)}")
            if tstFld:
                print(f"--- buildFill: StepE testfield {fillDict[tstFld]['value']}")


        # ── Step F: apply _FILL_MAP  (publish=True + resolve value) ───────
        for lk, spec in self._FILL_MAP.items():
            fid = lk_to_fid.get(lk)
            print("++++", lk, fid, spec)
            if not fid or fid not in fillDict:
                continue
            fd    = fillDict[fid]
            ftype = fd["fType"]
            

            if ftype in ("checkBox", "checkText", "box"):
                value = fd.get("checkedValue", "/1")
            else:
                value = self._resolve(spec["source"], spec["path"], src_map) or ""

            

            fillDict[fid].update({
                "publish": True,
                "source":  spec["source"],
                "path":    spec["path"],
                "note":    spec.get("note", ""),
                "value":   value,
            })

        return fillDict, lk
        
        if self.verbose:
            print(f"---- {self.oID} buildFillDict StepF verbose{self.verbose}, testFld:{tstFld}")
            if tstFld:
                print(f"-----buildFill: StepF testfield {fillDict[tstFld]}")
fillDict = xxx().foo(fObj)
fillDict['f20']

irsForm Entry: oID:Form1065, verbose:True
xxxx form1065 324 True
  ⚠️  Keys PDF not found for Form1065 — logicalKeys will be empty
  ⚠️  FieldNames JSON not found for Form1065 — labels will be empty
irsForm.Form1065 buildFillDict Entry


{'fID': 'f20',
 'pdfField': 'topmostSubform[0].Page1[0].f1_19[0]',
 'shortName': 'f1_19',
 'logicalKey': '',
 'label': '',
 'fType': 'text',
 'page': 1,
 'location': 'Form1065.Pg1.Unknown',
 'checkedValue': '/1',
 'publish': False,
 'source': None,
 'path': None,
 'note': '',
 'value': ''}

In [ ]:
{fd["logicalKey"]: fid  for fid, fd in fillDict.items()  if fd["logicalKey"]
        }

In [ ]:
lk_to_fid: Dict[str, str] = {
            fd["logicalKey"]: fid
            for fid, fd in fillDict.items()
            if fd["logicalKey"]
        }
lk_to_fid

irsForm Entry: oID:Form1065, verbose:True
xxxx form1065 324 True
  ⚠️  Keys PDF not found for Form1065 — logicalKeys will be empty
  ⚠️  FieldNames JSON not found for Form1065 — labels will be empty
irsForm.Form1065 buildFillDict Entry


{}

In [27]:
class aaa:
    def bbb(self):
        xxx : dict(a=1, b=2)
aaa().bbb().

AttributeError: 'NoneType' object has no attribute 'xxx'

In [3]:
# ── 1. Instantiate ────────────────────────────────────────────────
#    Option A: pass an LLC object (derives irsDir automatically)
#    f1065 = Form1065(llc=llc)
#
#    Option B: no llc — uses fallback path from __file__ location


from irs.Form1065 import Form1065
fObj = Form1065(llc, verbose=True)
fillDict_1065 = fObj._to_PDF(testField='f20')

from irs.Sch_K1 import Sch_K1
fillDict_SchK1 = Sch_K1(llc, verbose=True)._to_PDF()

from irs.Form4562 import Form4562
fillDict_4562 = Form4562(llc, verbose=True)._to_PDF()



    

irsForm Entry: oID:Form1065, verbose:True
xxxx form1065 324 True
irsForm.Form1065 _to_PDF Entry
---- irsForm._to_PDF Entry
  ⚠️  Keys PDF not found for Form1065 — logicalKeys will be empty
  ⚠️  FieldNames JSON not found for Form1065 — labels will be empty
nSpace Created dict_keys(['form', 'source', 'total_fields', 'fields'])
  ✅ namespace JSON  → Form1065_namespace.json  (500 fields)
  ✅ namespace PDF   → Form1065_namespace.pdf  (text=350, checkBox=90/90)
Form1065 buildFillDict Entry verbose:True, testField:f20
irsForm.Form1065 buildFillDict Entry
Acct.Rev List: 12, revList:7
Acct.Rev List: 12, revList:7
  ℹ️  llcFinancialReport loaded  (net_income=-2291.37, total_assets=215597.39)
--- Form1065 buildFillDict StepE verboseTrue, testFld:f20, is_data:13, bs_data:13
--- buildFill: StepE testfield 
---- Form1065 buildFillDict StepF verboseTrue, testFld:f20
-----buildFill: StepF testfield 
buildFill: Final testfield 
  ✅ fillDict built  → 440 fields total  (publish=0 [text=0, chk=0], cpa=0,

In [74]:
df = pd.DataFrame(fillDict_1065).transpose()
#df[df.publish is True]
df.value.unique()
fillDict_1065['f20']

{'fID': 'f20',
 'pdfField': 'topmostSubform[0].Page1[0].f1_19[0]',
 'shortName': 'f1_19',
 'logicalKey': '',
 'label': '',
 'fType': 'text',
 'page': 1,
 'location': 'Form1065.Pg1.Unknown',
 'checkedValue': '/1',
 'publish': False,
 'source': None,
 'path': None,
 'note': '',
 'value': ''}

In [57]:
pd.DataFrame(fillDict_SchK1).transpose()

,fID,pdfField,shortName,logicalKey,label,fType,page,location,checkedValue,publish,source,path,note,value
f2,f2,topmostSubform[0].Page1[0].Pg1Header[0].ForCal...,f1_1,,,text,1,Sch_K1.Pg1.Unknown,/1,False,None,None,,
f3,f3,topmostSubform[0].Page1[0].Pg1Header[0].ForCal...,f1_2,,,text,1,Sch_K1.Pg1.Unknown,/1,False,None,None,,
f4,f4,topmostSubform[0].Page1[0].Pg1Header[0].ForCal...,f1_3,,,text,1,Sch_K1.Pg1.Unknown,/1,False,None,None,,
f5,f5,topmostSubform[0].Page1[0].Pg1Header[0].ForCal...,f1_4,,,text,1,Sch_K1.Pg1.Unknown,/1,False,None,None,,
f6,f6,topmostSubform[0].Page1[0].Pg1Header[0].ForCal...,f1_5,,,text,1,Sch_K1.Pg1.Unknown,/1,False,None,None,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
f133,f133,topmostSubform[0].Page1[0].RightCol[0].RightCo...,Line15,,,text,1,Sch_K1.Pg1.Unknown,/1,False,None,None,,
f134,f134,topmostSubform[0].Page1[0].RightCol[0].RightCo...,Line17,,,text,1,Sch_K1.Pg1.Unknown,/1,False,None,None,,
f135,f135,topmostSubform[0].Page1[0].RightCol[0].RightCo...,Line18,,,text,1,Sch_K1.Pg1.Unknown,/1,False,None,None,,
f136,f136,topmostSubform[0].Page1[0].RightCol[0].RightCo...,Line19,,,text,1,Sch_K1.Pg1.Unknown,/1,False,None,None,,


In [56]:
pd.DataFrame(fillDict_4562).transpose()

,fID,pdfField,shortName,logicalKey,label,fType,page,location,checkedValue,publish,source,path,note,value
f2,f2,topmostSubform[0].Page1[0].f1_1[0],f1_1,,,text,1,Form4562.Pg1.Unknown,/1,False,None,None,,
f3,f3,topmostSubform[0].Page1[0].f1_2[0],f1_2,,,text,1,Form4562.Pg1.Unknown,/1,False,None,None,,
f4,f4,topmostSubform[0].Page1[0].f1_3[0],f1_3,,,text,1,Form4562.Pg1.Unknown,/1,False,None,None,,
f5,f5,topmostSubform[0].Page1[0].f1_4[0],f1_4,,,text,1,Form4562.Pg1.Unknown,/1,False,None,None,,
f6,f6,topmostSubform[0].Page1[0].f1_5[0],f1_5,,,text,1,Form4562.Pg1.Unknown,/1,False,None,None,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
f315,f315,topmostSubform[0].Page3[0].c3_3[1],c3_3,,,checkText,3,Form4562.Pg3.Unknown,/2,False,None,None,,
f316,f316,topmostSubform[0].Page3[0].c3_4[0],c3_4,,,checkBox,3,Form4562.Pg3.Unknown,/1,False,None,None,,
f317,f317,topmostSubform[0].Page3[0].c3_4[1],c3_4,,,checkText,3,Form4562.Pg3.Unknown,/2,False,None,None,,
f318,f318,topmostSubform[0].Page3[0].c3_5[0],c3_5,,,checkBox,3,Form4562.Pg3.Unknown,/1,False,None,None,,


In [34]:
self = f1065
from typing import Dict, List, Optional, Tuple
import re
is_data = None
bs_data = None

# ── Step A: base dict — all fields, publish=False, value="" ───────
fillDict = self._defaultFill(nspaceDict)

In [2]:
# LLC GL data
import os
from pathlib import Path
import json
from ledger.LLC import LLC
from irs.pdfFill import pdfFill
from IPython.display import display, Markdown

# Link to LLC ledgers
top = Path.cwd().parents[2]
llcName = [f for f in os.listdir(top) if 'llcProfile' in f][0].replace('.json','').replace('llcProfile_','')
llc = LLC(llcName,debug=False, top=top)
# Save entity information
eDict = llc.entity
acctDIR = llc.acctDir() #os.path.join(llc.TOP, llc.dirAccounting, str(llc.yr))
yeDIR = llc.acctDir(dirName='ye')


# ----------   initialize IRS forms
from irs.irsForms import irsF1065, irsSchK1
           
fmSchK1 = irsSchK1(llc)
fm1065 = irsF1065(llc)

fm = fmSchK1

## irsFormWorkFlow - form-IRS.pdf -> Form-keys.pdf

#### Input:  Form-FieldNames.json

- option: genFldNames=True| will generate (**replace**) Form-FieldNames.json


In [3]:
#irsFormWorkFlow - form-IRS.pdf -> Form-keys.pdf, using Form-FieldNames.json
def irsFormWorkFlow(fm, **kwargs):
    debug = kwargs.get('debug', False)

    
    
    display(Markdown(f"##### \n## Notebook using:\n### LLC: {llc.entity['entity_name']}\n\n### IRS Form: {fm.oID}"))
    inFN = fm.inFN()
    outFN = fm.outFN()
    keyFN = fm.keyFN()
    fldNmFN = fm.FN('FieldNames.json')
    print("<PDF IN     :", Path(inFN).name)
    print("<JSON FldNm :", Path(fldNmFN).name)
    print(">JSON Key   :", Path(outFN).name)
    print(">PDF OUT    :", Path(keyFN).name)
    
    
    ## Per Tax Year - Do once
    
    # Per Year, Save Form json field names, modify if any changes, json save per year
    # Do this once per year to reconcile new forms to IRS definitions.
    genFldNames = kwargs.get('genFldNames', False)
    if genFldNames: 
        if fm.oID == 'fmSchK1':
            from irs.irsFormFieldNames import irsSchKFields
            fm.fldNmSave(irsSchKFields().fldNmDict)
        elif fm.oID == 'irsF1065':
            from irs.irsFormFieldNames import irsF1065Fields
            fm1065.fldNmSave(irsF1065Fields().fldNmDict)
        print(f"Create key.json: {fm.oID}")
    
    ## Get Dict of all fields in form :: fm.genTestKeyDict()

    yeDIR = fm.llc.acctDir(to = 'ye')
    irsFormDir = os.path.join(yeDIR, 'Forms_IRS')
    
    #---- 1. Get form fields
    pf = pdfFill(inFN, outFN)
    fDict = pf.get()
    print(f"\nStep 1: Loaded form Field Name")
    if debug: 
        print(f"Sample fDict\n{list([(k,d) for k,d in testDict.items()])[0:3]}")
    
    # --- 2. Load Form map (json) : field index matches order offDict.keys()
    fldFN = fldNmFN
    fldNmDict = fm.fldNmLoad()
    # Load from py
    #fldNmDict = irsF1065Fields().fldNmDict
    print(f"\nStep 2: Load form Field Keys: key.fld:{len(fldNmDict)}/fm.fld{len(fDict)}, FILE: {fldFN}")
    if debug: 
        print(f"View fldNmDict\n{list([(k,d) for k,d in fldNmDict.items()])[0:3]}")
    
    # ---- 3. Create testDict to map field ID (F#) into every field
    #         OLD testDict = {k:self.pf._testKey(i,k,fDict[k]) for i,k in enumerate(fDict)}
    fldNmDict = fm.fldNmLoad()
    testDict =  {}
    for (i,k1),(j,k2) in zip(enumerate(fDict), enumerate(fldNmDict)):
        d = fDict[k1]
        d['field_id'] = k2
        testDict[k1] = d
    print(f"\nStep 3: Create Map: : testDict:{len(testDict)} -> keyNm:{len(fldNmDict)} x fm.fld:{len(fDict)}")
    if debug: 
        print(f"View testDict\n{list([(k,d) for k,d in testDict.items()])[0:3]}")
    
    
    # ---- 4. Output: PDF key.pdf  
    ts = pdfFill(inFN, outFN)
    oDict = ts.fillPDF(to = testDict, )
    print(f"\nStep 4: Create outPDF") 
    
    print(f"\n{fm.__class__.__name__} SUCC: testDict Generated {len(testDict)} Fields; \n-- FILE: {Path(keyFN)}")

irsFormWorkFlow(fm1065)  # genFldNames=True|False
print("\n\n", '*'*60, "\n\n")
irsFormWorkFlow(fmSchK1)  # genFldNames=True|False
     
    
      
    

##### 
## Notebook using:
### LLC: W&B Group, LLC

### IRS Form: irsF1065

<PDF IN     : Form_1065-IRS.pdf
<JSON FldNm : Form_1065-FieldNames.json
>JSON Key   : Form_1065-keys.pdf
>PDF OUT    : Form_1065-keys.json

Step 1: Loaded form Field Name
Loading Form Field Name (json): Form_1065-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Form_1065-FieldNames.json

Step 2: Load form Field Keys: key.fld:440/fm.fld440, FILE: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Form_1065-FieldNames.json
Loading Form Field Name (json): Form_1065-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Form_1065-FieldNames.json

Step 3: Create Map: : testDict:440 -> keyNm:440 x fm.fld:440

Step 4: Create outPDF

irsF1065 SUCC: testDict Generated 440 Fields; 
-- FILE: /Users/frankrojas/GDrive/Famil

##### 
## Notebook using:
### LLC: W&B Group, LLC

### IRS Form: irsSchK1

<PDF IN     : Schedule_K_1-IRS.pdf
<JSON FldNm : Schedule_K_1-FieldNames.json
>JSON Key   : Schedule_K_1-keys.pdf
>PDF OUT    : Schedule_K_1-keys.json

Step 1: Loaded form Field Name
Loading Form Field Name (json): Schedule_K_1-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-FieldNames.json

Step 2: Load form Field Keys: key.fld:111/fm.fld111, FILE: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-FieldNames.json
Loading Form Field Name (json): Schedule_K_1-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-FieldNames.json

Step 3: Create Map: : testDict:111 -> keyNm:111 x fm.fld:111

Step 4: Create outPDF

irsSchK1 SUCC: testDict Generated 111 Fields; 
-- FILE: /Us

In [ ]:
print(len(glDict), len(testDict))
d = {}
for (i,k1),(j,k2)  in  zip(enumerate(glDict), enumerate(testDict)):
    g = glDict[k1]
    fldDict = testDict[k2]
    f = fldDict['field_id']
    if f != g : 
        print("ERROR: glDict and fmDict have a mismatch")    
        print("--ERR>", i,j, glDict[k1], testDict[k2])
    v = '' if fldDict['type'] == 'text' else 'chk'
    d[k1] = v
list([(k,d) for k,d in d.items()])[0:3]
fn = fm.FN('glKeys.json')
print(fn)
with open(fn, 'w') as fio:
    json.dump(d, fio, indent=4)